# Comparing reconditioning procedures against inversion routines

**Exploratory. Informs the choice of treatments carried into Chapter 5.**

Before any treatment was tested on the genuine Bayesian pipeline, this notebook surveys the field
on simulated data: eight ways of reconditioning a covariance crossed with four ways of inverting
it, each scored across 500 matrices.

The purpose is to narrow the field cheaply. A treatment which fails here need not be carried
forward; a treatment which succeeds here still has to be confirmed on the real fit, since a
least-squares proxy is not the Bayesian estimator, and every figure reported in the dissertation
was obtained afterwards on the genuine pipeline.

Two results determined what followed: the inversion routine matters far less than the
reconditioning, and the apparent success of non-linear shrinkage was an artefact of reporting the
smallest eigenvalue rather than the condition number.

### Imports

Everything the notebook uses, gathered here so that no later cell imports anything of its own.

In [30]:
import numpy as np
from scipy.linalg import pinvh
from pathlib import Path

### The simulated data

A package of 500 covariance matrices, each 126×126, built from independent random walks of known diffusion coefficient. `cov` holds the matrices, `msd` the mean-squared displacement curves they were estimated from.

In [31]:
base = Path.home() / "Desktop" / "Dissertation" / "kappa-kinisi"

data = np.load(base / "data" / "kinisi_rw_data_1.npz")

cov = data["cov"]

msd = data["msd"]

timestep = msd[0, :, 0]

n_samples = msd[0, :, 3]

dim = 3

d_true = 1.0

The covariance is the object under test; the MSD curves supply the observations each treated matrix will be asked to fit.

### The analytical covariance

The noise-free matrix the sampled ones are compared against. Its smallest eigenvalue is the target every reconditioner is trying to approach.

In [32]:
design = np.column_stack([timestep, np.ones_like(timestep)])

n = n_samples.shape[0]

analytical = np.zeros((n, n))

for i in range(n):
    for j in range(i, n):
        analytical[i, j] = 8 * dim**2 * timestep[i]**2 / (dim * n_samples[j])

        analytical[j, i] = analytical[i, j]

true_lmin = np.linalg.eigvalsh(analytical)[0]

print("true smallest eigenvalue:", round(true_lmin, 4))

true smallest eigenvalue: 0.003


The analytical matrix is positive definite, so its smallest eigenvalue is a small positive number. A reconditioner succeeds insofar as it brings a damaged matrix back towards this value without distorting the rest of the spectrum.

### The eight reconditioning procedures

Each takes a covariance and returns a treated one. They range from doing nothing, through the
floors and ridges that raise the small eigenvalues, to the two shrinkage methods that pull the
spectrum towards a target. Two of them, `mineig 1e16` and `clip_mp`, are what kinisi applies on
its default and optional paths respectively.

### The inversion routines

The pseudo-inverse is the one the package uses. Note that it discards directions below a numerical tolerance without warning, which is examined in Section 5.7.1.

In [33]:
def r_none(m):
    """
    Return the matrix unchanged, as the baseline for comparison.
    
    """
    return m




def r_mineig(m, kappa=1e16):
    """
    Raise every eigenvalue to a floor set by a target condition number.

    The floor is the largest eigenvalue divided by kappa, so no eigenvalue is
    left smaller than that. This is the treatment kinisi applies on its default
    path, reproduced here for comparison; the default kappa of 1e16 is far below
    anything the data produces, so on a healthy matrix it does almost nothing.

    """
    v, V = np.linalg.eigh(m)
    floor = v[-1] / kappa
    return V @ np.diag(np.maximum(v, floor)) @ V.T




def r_mineig_k3(m):
    """
    The minimum-eigenvalue floor at an aggressive target of 1e3.

    Included to show that driving the condition number down hard gives the
    best-conditioned matrices and among the worst recovery: a matrix can be made
    numerically respectable and less useful at the same time.

    """
    return r_mineig(m, 1e3)




def r_ridge(m, kappa=1e3):
    """
    Add a constant to the diagonal until the condition number reaches kappa.

    The constant delta shifts every eigenvalue up by the same amount, which
    lifts the smallest without touching the eigenvectors. A blunt instrument: it
    penalises the well-determined directions as heavily as the damaged ones.

    """
    v = np.linalg.eigvalsh(m)
    delta = (v[-1] - kappa * v[0]) / (kappa - 1)
    return m + delta * np.eye(m.shape[0])




def r_adaptive(m, c=0.25):
    """
    Lift only the negative eigenvalues, to a floor set by the damage present.

    Where the smallest eigenvalue is negative, any eigenvalue below c times its
    magnitude is raised to that value; a positive-definite matrix is left
    untouched. Unlike the fixed floors above, the threshold scales with how
    damaged the matrix is, which is the property that motivated it. This is the
    adaptive floor developed in this work; the coefficient is justified in
    realfit_c_sweep.py.

    """
    v, V = np.linalg.eigh(m)
    if v[0] < 0:
        floor = c * abs(v[0])
        v = np.maximum(v, floor)
    return V @ np.diag(v) @ V.T




def r_clip_mp(m):
    """
    Raise eigenvalues to a bound derived from the Marcenko-Pastur law.

    The bound is the upper edge of the eigenvalue distribution expected for a
    sample covariance of this shape. Note that this floors the small eigenvalues
    up to the bound rather than removing them, so it is a floor with a
    principled threshold rather than true spectral clipping; it is included to
    test whether an MP-derived threshold improves on the fixed floors above.

    """
    v, V = np.linalg.eigh(m)
    q = m.shape[0] / max(n_samples)
    sigma2 = np.median(v) / (1 + np.sqrt(q))**2
    bound = sigma2 * (1 + np.sqrt(q))**2
    return V @ np.diag(np.maximum(v, bound)) @ V.T




def r_nls_analytical(m):
    """
    Blend the spectrum halfway towards the analytical eigenvalues.

    An oracle method: it uses the analytical covariance, which is not available
    for real data, and so cannot be used in practice. It is included as a bound
    on what shrinkage towards a known target could achieve. It also illustrates
    a trap, since it can restore a positive smallest eigenvalue while leaving
    the condition number enormous.

    """
    v, V = np.linalg.eigh(m)
    target = np.linalg.eigvalsh(analytical)
    blended = 0.5 * v + 0.5 * target
    return V @ np.diag(blended) @ V.T




def r_taper(m, c=0.5):
    """
    Soften the small eigenvalues smoothly rather than flooring them.

    Each eigenvalue is replaced by a smooth function that leaves the large ones
    essentially unchanged and lifts the small ones towards a scale set by c
    times the largest. Unlike a floor, the transition is gradual, with no sharp
    threshold at which behaviour changes.

    """
    v, V = np.linalg.eigh(m)
    scale = v[-1]
    soft = 0.5 * (v + np.sqrt(v**2 + (c * scale)**2))
    return V @ np.diag(soft) @ V.T




reconditioners = {
    "none": r_none,
    "mineig 1e16": r_mineig,
    "mineig 1e3": r_mineig_k3,
    "ridge": r_ridge,
    "adaptive": r_adaptive,
    "clip MP": r_clip_mp,
    "NLS analytic": r_nls_analytical,
    "taper": r_taper,
}

### The four inversion routines

The pseudo-inverse is the one kinisi uses. It is included because it discards any direction whose eigenvalue falls below a numerical tolerance, which turns out to matter more than any choice of reconditioner.

In [34]:
def i_plain(m):
    return np.linalg.inv(m)


def i_pinvh(m):
    return pinvh(m)


def i_tikhonov(m, lam=1e-6):
    return np.linalg.inv(m + lam * np.eye(m.shape[0]))


def i_tsvd(m, k=120):
    v, V = np.linalg.eigh(m)
    keep = np.zeros_like(v)
    keep[-k:] = 1.0 / v[-k:]
    return V @ np.diag(keep) @ V.T


inverses = {
    "plain": i_plain,
    "pinvh": i_pinvh,
    "tikhonov": i_tikhonov,
    "tsvd": i_tsvd,
}

### The fit

A generalised least squares slope, weighted by the inverse covariance. This is the least-squares proxy for the Bayesian fit: quicker to run across 500 matrices, but not the estimator the dissertation reports.

In [36]:
def gls_D(sigma_inv, y):
    lhs = A.T @ sigma_inv @ A
    rhs = A.T @ sigma_inv @ y
    try:
        beta = np.linalg.solve(lhs, rhs)
        return beta[0] / (2 * dim)
    except np.linalg.LinAlgError:
        return np.nan

In [37]:
def gls_slope(sigma_inv, y):
    """
    Compute the diffusion coefficient from a generalized least-squares fit.

    Returns NaN when the GLS normal equations are singular or cannot be
    solved numerically.
    """

    lhs = design.T @ sigma_inv @ design

    rhs = design.T @ sigma_inv @ y

    gradient = np.linalg.solve(lhs, rhs)[0]

    return gradient / (2 * dim)

Dividing the gradient by $2d$ converts it to a diffusion coefficient, so a correct fit returns a value near the true one of unity.

### The survey

Every reconditioner crossed with every inverse, scored across 500 matrices on the error in the smallest eigenvalue, the resulting condition number, and the accuracy and bias of the recovered diffusion coefficient.

In [38]:
n_sim = 500

results = {}

for rname, rfn in reconditioners.items():
    treated = rfn(cov[0])

    ev = np.linalg.eigvalsh(treated)

    lmin = ev[0]

    kappa = ev[-1] / lmin if lmin > 0 else np.inf

    lmin_err = abs(lmin - true_lmin)




    for iname, ifn in inverses.items():
        recovered = []

        for k in range(n_sim):
            m = rfn(cov[k])
            try:
                recovered.append(gls_slope(ifn(m), msd[k, :, 1]))
            except (np.linalg.LinAlgError, ValueError):
                recovered.append(np.nan)


        recovered = np.array(recovered)

        good = recovered[np.isfinite(recovered)]

        good = good[(good > -5) & (good < 5)]


        results[(rname, iname)] = {
            "lmin_err": lmin_err,
            "log_kappa": np.log10(kappa) if np.isfinite(kappa) else 16.0,
            "rmse": np.sqrt(np.mean((good - d_true)**2)) if len(good) else np.nan,
            "bias": np.median(good) - d_true if len(good) else np.nan,
        }

### The results table

Sorted by recovery error, best first.

In [39]:
print("reconditioning x inverse       lmin_err   logK    rmse      bias")

for (r, i), v in sorted(results.items(),
                        key=lambda x: np.nan_to_num(x[1]["rmse"], nan=1e9)):
    print(f"{r + ' x ' + i:28s} {v['lmin_err']:9.3f} {v['log_kappa']:6.2f} "
          f"{v['rmse']:9.4f} {v['bias']:+8.4f}")

reconditioning x inverse       lmin_err   logK    rmse      bias
none x tikhonov                 33.567  16.00    0.0146  -0.0024
none x pinvh                    33.567  16.00    0.0146  -0.0024
none x plain                    33.567  16.00    0.0146  -0.0024
mineig 1e16 x pinvh              0.003  16.01    0.0146  -0.0024
NLS analytic x plain            16.784  16.00    0.0160  -0.0015
NLS analytic x pinvh            16.784  16.00    0.0160  -0.0015
NLS analytic x tikhonov         16.784  16.00    0.0160  -0.0015
none x tsvd                     33.567  16.00    0.0194  -0.0034
NLS analytic x tsvd             16.784  16.00    0.0204  -0.0035
adaptive x tikhonov              8.388   4.02    0.0306  -0.0071
adaptive x plain                 8.388   4.02    0.0306  -0.0071
adaptive x pinvh                 8.388   4.02    0.0306  -0.0071
adaptive x tsvd                  8.388   4.02    0.0328  -0.0082
mineig 1e3 x plain              88.834   3.00    0.0407  -0.0097
mineig 1e3 x pinvh       

Three things follow from this table, and each shaped the rest of the project.

**The inverse barely matters.** For any given reconditioner the four inversion routines give almost
identical recovery, so the search narrowed to the treatment rather than the solver.

**Aggressive conditioning costs accuracy.** Driving the condition number down to $10^3$ gives the
best-conditioned matrices in the table and among the worst recovery, the first sign that a matrix
can be made numerically respectable and less useful at the same time.

**A minimum sits at intermediate thresholds.** Neither the most permissive nor the most aggressive
cap performs best, which suggested a threshold expressed relative to the damage present rather than
as a fixed target. That is the adaptive floor.

### Why the non-linear shrinkage result is not what it seems

The NLS row sits near the top of the table, which looks like a success. It is not. The single
matrix examined below shows why: the treatment reports a healthy smallest eigenvalue while leaving
the condition number at $10^{15}$, so it has restored positive definiteness without restoring
invertibility. Reporting the smallest eigenvalue alone would have concealed this, which is the
trap the survey was designed to expose.

In [40]:
m = cov[0]

raw_ev = np.linalg.eigvalsh(m)

nls_ev = np.linalg.eigvalsh(r_nls_analytical(m))

adaptive_ev = np.linalg.eigvalsh(r_adaptive(m))

print("raw:      lmin", f"{raw_ev[0]:+.4f}", "  condition", f"{raw_ev[-1] / abs(raw_ev[0]):.2e}")

print("NLS:      lmin", f"{nls_ev[0]:+.4f}", "  condition", f"{nls_ev[-1] / nls_ev[0]:.2e}")

print("adaptive: lmin", f"{adaptive_ev[0]:+.4f}", "  condition", f"{adaptive_ev[-1] / adaptive_ev[0]:.2e}")

raw:      lmin -33.5642   condition 2.65e+03
NLS:      lmin -16.7806   condition -5.22e+03
adaptive: lmin +8.3910   condition 1.06e+04


The distinction that runs through the whole project appears here in miniature. The non-linear
shrinkage makes the smallest eigenvalue positive but leaves the condition number enormous; the
adaptive floor lifts the small eigenvalues to a common value and brings the condition number down
by many orders of magnitude. A treatment must be judged on what it does to the whole spectrum, not
on the sign of one eigenvalue.

### Outcome

Two treatments were carried forward to the genuine pipeline: the adaptive floor, developed from
the minimum in the sweep, and Marčenko–Pastur clipping, because it is what kinisi applies when its
optional reconditioning is requested. Everything reported in Chapter 5 was obtained on the real
Bayesian fit, not on the proxy used here.